# Analiza wycieków informacji (data leakage)

Cel: zlokalizować w surowych danych artefakty redakcyjne i metadane, które jednoznacznie identyfikują klasę (Reuters / Fake) zamiast nieść sygnał *treściowy*. Wnioski zasilą listę wzorców w `src/preprocessing/cleaning.py` — bez tego cleaningu model uczy się stempla wydawcy, nie języka fake-newsa.

Pracujemy na **surowych** `True.csv` / `Fake.csv` (przed cleaningiem).

In [1]:
import re
import warnings
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import chi2, mutual_info_classif

warnings.filterwarnings('ignore', category=UserWarning)
pd.options.plotting.backend = 'plotly'
pd.set_option('display.max_colwidth', 120)

LABEL_NAMES = {0: 'Fake', 1: 'Real'}
LABEL_COLORS = {'Fake': '#EF553B', 'Real': '#636EFA'}
RNG = np.random.default_rng(42)

In [2]:
raw_dir = Path(kagglehub.dataset_download('clmentbisaillon/fake-and-real-news-dataset'))
true_df = pd.read_csv(raw_dir / 'True.csv').assign(label=1)
fake_df = pd.read_csv(raw_dir / 'Fake.csv').assign(label=0)
df = pd.concat([true_df, fake_df], ignore_index=True)
df['label_name'] = df['label'].map(LABEL_NAMES)
print(f'Razem: {len(df):,} | Real: {(df.label==1).sum():,} | Fake: {(df.label==0).sum():,}')
df.head(2)

Razem: 44,898 | Real: 21,417 | Fake: 23,481


,title,text,subject,date,label,label_name
0,"As U.S. budget fight looms, Republicans flip their fiscal script","WASHINGTON (Reuters) - The head of a conservative Republican faction in the U.S. Congress, who voted this month for ...",politicsNews,"December 31, 2017",1,Real
1,U.S. military to accept transgender recruits on Monday: Pentagon,WASHINGTON (Reuters) - Transgender people will be allowed for the first time to enlist in the U.S. military starting...,politicsNews,"December 29, 2017",1,Real


## 1. `subject` jako bezpośrednia etykieta

Pierwszy podejrzany sygnał: kolumna `subject`. Jeśli któraś wartość występuje wyłącznie w jednej klasie, to nie jest cecha — to **etykieta pod inną nazwą**.

In [3]:
subject_xtab = pd.crosstab(df['subject'], df['label_name'], margins=True, margins_name='Razem')
subject_xtab

label_name,Fake,Real,Razem
subject,,,
Government News,1570,0,1570
Middle-east,778,0,778
News,9050,0,9050
US_News,783,0,783
left-news,4459,0,4459
politics,6841,0,6841
politicsNews,0,11272,11272
worldnews,0,10145,10145
Razem,23481,21417,44898


In [4]:
subj_share = pd.crosstab(df['subject'], df['label_name'], normalize='index').round(3)
fig = px.bar(subj_share.reset_index().melt(id_vars='subject', var_name='Klasa', value_name='Udział'),
             x='subject', y='Udział', color='Klasa',
             color_discrete_map=LABEL_COLORS,
             title='Udział klas w obrębie każdej wartości <b>subject</b>')
fig.update_layout(xaxis_title='subject', yaxis_title='udział klasy', barmode='stack',
                  height=400, legend_title='')
fig.show()

**Wniosek (1).** Każda wartość `subject` jest w 100% przypisana do dokładnie jednej klasy (`politicsNews` i `worldnews` → Real; reszta → Fake). Mamy więc **idealny wyciek** — jedna kolumna kategoryczna jednoznacznie wyznacza target. `subject` *musi* zostać usunięty z wejścia modelu.

## 2. Markery redakcyjne w `text`

Zdefiniujmy regexowe markery typowych artefaktów (źródło: ręczny przegląd próbek + analiza w `notebooks/explore_temp/03_leakage_analysis.ipynb`). Dla każdego markera tworzymy cechę binarną — obecność w danym dokumencie.

In [5]:
MARKERS = {
    'reuters_dateline': r'^[A-Z][A-Z\s/]{1,40} \(Reuters\)\s*-',
    'reuters_word':     r'\breuters\b',
    'url_http':         r'https?://\S+',
    'twitter_pic':      r'pic\.twitter\.com',
    'at_handle':        r'@\w+',
    'hashtag':          r'#\w+',
    'getty_images':     r'getty\s+images?',
    'featured_image':   r'featured\s+image',
    'wire_21cw':        r'21st\s+century\s+wire',
    'video_marker':     r'\b(?:VIDEO|WATCH|BREAKING)\s*:',
    'via_handle':       r'\bvia\s+@\w+',
}

marker_df = pd.DataFrame({'label': df['label'], 'label_name': df['label_name']})
for name, pat in MARKERS.items():
    marker_df[name] = df['text'].str.contains(pat, regex=True, flags=re.IGNORECASE).fillna(False)
marker_df.head(3)

,label,label_name,reuters_dateline,reuters_word,url_http,twitter_pic,at_handle,hashtag,getty_images,featured_image,wire_21cw,video_marker,via_handle
0,1,Real,True,True,False,False,False,False,False,False,False,False,False
1,1,Real,True,True,False,False,False,False,False,False,False,False,False
2,1,Real,True,True,False,False,False,False,False,False,False,False,False


In [6]:
marker_cols = list(MARKERS.keys())
presence = (marker_df.groupby('label_name')[marker_cols].mean() * 100).round(2).T
presence['delta_pp'] = (presence['Real'] - presence['Fake']).round(2)
presence.sort_values('delta_pp', key=np.abs, ascending=False)

label_name,Fake,Real,delta_pp
reuters_word,1.31,99.82,98.51
reuters_dateline,0.00,83.21,83.21
featured_image,34.76,0.00,-34.76
at_handle,26.06,1.31,-24.75
getty_images,16.82,0.00,-16.82
twitter_pic,14.79,0.00,-14.79
url_http,14.05,0.00,-14.05
hashtag,13.28,1.11,-12.17
video_marker,5.49,0.02,-5.47
wire_21cw,5.34,0.00,-5.34


In [7]:
plot_df = (presence[['Fake', 'Real']]
           .reset_index()
           .melt(id_vars='index', var_name='Klasa', value_name='% dokumentów'))
fig = px.bar(plot_df, x='index', y='% dokumentów', color='Klasa', barmode='group',
             color_discrete_map=LABEL_COLORS,
             title='Obecność markerów redakcyjnych w surowym <b>text</b> (% dokumentów per klasa)')
fig.update_layout(xaxis_title='marker', yaxis_title='% dokumentów',
                  xaxis_tickangle=-30, height=440, legend_title='')
fig.show()

**Wniosek (2).** Markery dzielą się na trzy grupy:
- **Real-only** — `reuters_dateline`, `reuters_word` występują niemal wyłącznie w Real (>90% vs ~0% w Fake). Każdy artykuł Reutersa zaczyna się od stempla `WASHINGTON (Reuters) - …`.
- **Fake-only** — `url_http`, `twitter_pic`, `at_handle`, `getty_images`, `featured_image`, `video_marker`, `via_handle`, `wire_21cw` występują głównie w Fake (kilka–kilkadziesiąt %, w Real ≪1%). To artefakty stylu blogowego, których Reuters nie używa.
- **Słabsze** — `hashtag` jest jedynie lekko leaky.

Każdy z tych wzorców *sam w sobie* wystarczyłby do trywialnej klasyfikacji — kasujemy je w cleaningu.

## 3. Test dyskryminacyjny: chi² i Mutual Information

Sformalizujmy obserwację z bloku 2 — dla każdej cechy binarnej liczymy chi² (siła związku z klasą) i wzajemną informację (ile bitów o klasie niesie sama obecność markera).

In [8]:
X = marker_df[marker_cols].astype(int).values
y = marker_df['label'].values

chi2_stat, p_val = chi2(X, y)
mi = mutual_info_classif(X, y, discrete_features=True, random_state=42)

scores = (pd.DataFrame({
        'marker': marker_cols,
        'chi2': chi2_stat,
        'p_value': p_val,
        'mutual_info': mi,
    })
    .sort_values('mutual_info', ascending=False)
    .reset_index(drop=True))
scores.assign(chi2=lambda d: d.chi2.round(0), mutual_info=lambda d: d.mutual_info.round(4))

,marker,chi2,p_value,mutual_info
0,reuters_word,22502.0,0.000000e+00,0.6497
1,reuters_dateline,19540.0,0.000000e+00,0.4559
2,featured_image,7444.0,0.000000e+00,0.1363
3,at_handle,4811.0,0.000000e+00,0.0762
4,getty_images,3603.0,0.000000e+00,0.0609
5,twitter_pic,3169.0,0.000000e+00,0.0531
6,url_http,3009.0,0.000000e+00,0.0503
7,hashtag,2221.0,0.000000e+00,0.0318
8,wire_21cw,1144.0,1.012480e-250,0.0185
9,video_marker,1162.0,1.039323e-254,0.0184


In [9]:
fig = px.bar(scores.sort_values('mutual_info'), x='mutual_info', y='marker', orientation='h',
             title='Mutual Information markerów z etykietą (im wyżej, tym silniejszy wyciek)')
fig.update_layout(xaxis_title='MI [nat]', yaxis_title='', height=440)
fig.show()

**Wniosek (3).** Wszystkie p-wartości chi² są praktycznie zerowe — każdy z markerów jest istotnie skorelowany z klasą. MI dominuje `reuters_dateline` (~0.4 nat), bo jest binarnym, niemal idealnym predyktorem — sama obecność stempla daje >0.5 bita informacji o klasie. Dla porównania, MI sensownych markerów językowych w klasyfikacji tekstu rzadko przekracza 0.05.

## 4. Top n-gramów per klasa (TF-IDF)

Spojrzenie szersze niż ręcznie dobrane regexy: jakie tokeny i bigramy są najbardziej klasowo-specyficzne? Liczymy średni TF-IDF per klasa i sortujemy po różnicy `Real − Fake`.

In [10]:
vec = TfidfVectorizer(
    lowercase=True, ngram_range=(1, 2),
    min_df=20, max_features=20000,
    token_pattern=r'(?u)\b[a-zA-Z][a-zA-Z]+\b',
)
X_tfidf = vec.fit_transform(df['text'].fillna(''))
vocab = np.array(vec.get_feature_names_out())

mask_real = (df['label'] == 1).values
mean_real = np.asarray(X_tfidf[mask_real].mean(axis=0)).ravel()
mean_fake = np.asarray(X_tfidf[~mask_real].mean(axis=0)).ravel()
diff = mean_real - mean_fake

top_real = pd.DataFrame({'ngram': vocab, 'tfidf_real': mean_real,
                          'tfidf_fake': mean_fake, 'delta': diff}
             ).nlargest(25, 'delta').reset_index(drop=True)
top_fake = pd.DataFrame({'ngram': vocab, 'tfidf_real': mean_real,
                          'tfidf_fake': mean_fake, 'delta': diff}
             ).nsmallest(25, 'delta').reset_index(drop=True)
f'TF-IDF macierz: {X_tfidf.shape}, słownik: {len(vocab):,}'

'TF-IDF macierz: (44898, 20000), słownik: 20,000'

In [11]:
print('TOP 25 n-gramów charakterystycznych dla Real:')
display(top_real.round(4))
print('TOP 25 n-gramów charakterystycznych dla Fake:')
display(top_fake.round(4))

TOP 25 n-gramów charakterystycznych dla Real:


,ngram,tfidf_real,tfidf_fake,delta
0,said,0.0502,0.0143,0.0359
1,reuters,0.0243,0.0002,0.0241
2,on,0.0471,0.0288,0.0183
3,in,0.0710,0.0554,0.0156
4,said on,0.0144,0.0005,0.0140
5,washington reuters,0.0110,0.0000,0.0110
6,minister,0.0119,0.0009,0.0110
7,its,0.0163,0.0053,0.0110
8,the,0.1799,0.1692,0.0107
9,china,0.0116,0.0013,0.0103


TOP 25 n-gramów charakterystycznych dla Fake:


,ngram,tfidf_real,tfidf_fake,delta
0,you,0.0040,0.0280,-0.0240
1,is,0.0237,0.0430,-0.0193
2,that,0.0364,0.0535,-0.0172
3,this,0.0110,0.0274,-0.0164
4,trump,0.0331,0.0476,-0.0145
5,her,0.0086,0.0213,-0.0127
6,hillary,0.0029,0.0147,-0.0118
7,she,0.0094,0.0209,-0.0114
8,just,0.0031,0.0144,-0.0113
9,it,0.0223,0.0335,-0.0112


In [12]:
combined = pd.concat([
    top_real.assign(klasa='Real'),
    top_fake.assign(klasa='Fake', delta=lambda d: d.delta.abs()),
]).head(50)
fig = px.bar(combined, y='ngram', x='delta', color='klasa', orientation='h',
             color_discrete_map=LABEL_COLORS, facet_col='klasa',
             title='Najbardziej klasowo-specyficzne n-gramy (|TF-IDF Real − TF-IDF Fake|)')
fig.update_yaxes(matches=None, showticklabels=True)
fig.update_layout(height=720, showlegend=False)
fig.show()

**Wniosek (4).** Top n-gramy Reala są zdominowane przez **artefakty redakcyjne**: lokalizacje stemplów (`washington`, `reuters`, `said`, `wednesday`, `tuesday`, `said statement`), nazwy agencji i typowy język depesz. Top Fake to w dużej mierze tokeny z mediów społecznościowych (`pic`, `twitter`, `featured image`, `getty images`, `via`, `donald trump`) — czyli dokładnie te same wzorce co w bloku 2, tylko widziane automatycznie. Po cleaningu (usunięcie stempli, URL-i, podpisów obrazków) ranking ten powinien być zdominowany przez **treściowe** różnice słownictwa, nie redakcyjne.

## 5. Markery w `title` (kontekstowo, nie wchodzi do modelu)

Dla kompletności — tytuły też niosą wyciek. Skoro `title` i tak nie wchodzi do modelu (ustalone w EDA), to jest to potwierdzenie tej decyzji.

In [13]:
title_markers = {
    'all_caps_word': r'\b[A-Z]{3,}\b',
    'video_prefix':  r'^(?:VIDEO|WATCH|BREAKING|SHOCK|UNREAL|WOW)\s*[:!]',
    'colon_label':   r'\w+\s*:\s+\w',
    'starts_lower':  r'^[a-z]',
    'has_question':  r'\?',
}
title_df = pd.DataFrame({'label_name': df['label_name']})
for name, pat in title_markers.items():
    title_df[name] = df['title'].fillna('').str.contains(pat, regex=True)
(title_df.groupby('label_name').mean() * 100).round(2).T

label_name,Fake,Real
all_caps_word,83.19,8.82
video_prefix,5.89,0.00
colon_label,25.61,22.70
starts_lower,0.04,0.00
has_question,7.00,0.64


**Wniosek (5).** Tytuły Fake częściej zawierają ALL-CAPS, prefiksy typu `VIDEO:` / `WATCH:` i znaki zapytania (clickbait). Tytuły Real są stylistycznie spójne (Title Case, bez ALL-CAPS). Potwierdza to wcześniejszą decyzję o **nie używaniu `title`** jako wejścia modelu — uczyłby się stylu nagłówków, nie treści.

## Podsumowanie — lista wzorców dla pipeline'u cleaningu

**Już obecne w `src/preprocessing/cleaning.py`** (zweryfikowane przez przegląd kodu):
- `^[A-Z][A-Z\s/]+ \(Reuters\) -` — stempel agencji
- `\b(Reuters|AP|AFP)\b` w nawiasach + `\breuters\b`
- `https?://`, `www\.`, `pic\.twitter\.com`, `tmsnrt\.rs/` — URL-e
- `@\w+` — twitter handles
- `featured image via … (getty|flickr|shutterstock|ap)`, `getty images?` — podpisy obrazków
- `21st century wire` — sygnatura strony
- HTML tagi i encje
- pojedyncze litery, nadmiar białych znaków, lower-case na końcu

**Do rozważenia (z analizy powyżej):**
- prefiksy clickbaitowe `^(VIDEO|WATCH|BREAKING|SHOCK)\s*[:!]` — występują głównie w `title`, ale czasem trafiają do `text` po skopiowaniu nagłówka (marker `video_marker` ma niezerowy MI).
- `\bvia\s+@\w+` — domknie sytuacje gdy `@` zostanie usunięte, ale `via` zostaje sierotą; 
  obecnie usuwamy tylko `@\w+`, więc `via` przeżywa.
- nazwy znanych portali Fake (`Breitbart`, `InfoWars` itd.) — celowo NIE usuwamy, bo to legalny sygnał stylistyczny, nie czysty stempel redakcyjny.

**Decyzje wynikowe (potwierdzone, do `notebooks/report/00_raport.ipynb`):**
- `subject` — drop (1:1 z label, perfekcyjny wyciek).
- `title` — drop (clickbaitowe markery, ALL-CAPS, prefiksy).
- `date` — drop (nieparsowalne dla Fake, formatowo różne; analiza w `01_eda.ipynb` blok 3).
- Pipeline cleaningu w `03_cleaning.ipynb` musi usunąć wszystkie markery z bloku 2 *zanim* uruchomimy tokenizację. Sanity check w `04_post_cleaning.ipynb`.